# Banking AI-Agent — Colab runner

End-to-end runner for the Lab 3 banking agentic workflow. This single notebook:

1. Installs and starts **Ollama** with `gpt-oss:20b`.
2. Clones the project repo and installs dependencies (FastAPI + Unsloth).
3. Mounts Google Drive and extracts the **fine-tuned intent artifacts** from Lab 2.
4. Smoke-tests the intent classifier.
5. Launches the **FastAPI orchestrator** on port `8000`.
6. Exposes that port publicly via **Pinggy**.
7. Demos the workflow end-to-end on `examples/sample_requests.json`.

**Runtime:** Colab GPU (T4 16GB is the minimum; T4×2 or L4 is recommended).

## Cell 1 — Install + start Ollama

Adapted from the `Ollama-Pinggy.ipynb` reference. Ollama is launched in a background thread inside this Colab kernel; we will reach it from FastAPI over `localhost:11434`.

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, threading, time

ollama_env = os.environ.copy()
ollama_env['OLLAMA_HOST'] = '0.0.0.0'
ollama_env['OLLAMA_ORIGINS'] = '*'

def _serve():
    subprocess.Popen(['ollama', 'serve'], env=ollama_env)

threading.Thread(target=_serve, daemon=True).start()
time.sleep(6)
print('Ollama serve started in background.')

In [ ]:
# Pull the LLM used by the response-drafting node.
# Fallback to gpt-oss:7b if VRAM is tight after loading the intent model.
OLLAMA_MODEL = 'gpt-oss:20b'  # change to 'gpt-oss:7b' if you hit OOM
!ollama pull {OLLAMA_MODEL}
!curl -s http://localhost:11434/api/tags | head -c 400

## Cell 2 — Clone the repo and install Python deps

The intent model uses Unsloth, which has its own installer.

In [ ]:
%cd /content
![ -d banking-ai-agent ] || git clone https://github.com/tzin1401/banking-ai-agent.git
%cd /content/banking-ai-agent

In [ ]:
!pip install -q -r requirements.txt
# Unsloth has its own install command on Colab:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## Cell 3 — Mount Drive and extract the intent artifacts

The Lab 2 fine-tuned LoRA adapter lives in
`MyDrive/banking-ai-agent/intent_artifacts.zip` (~347 MB). After extraction the
project root contains `outputs/`, `sample_data/`, and `configs/inference.yaml`,
which is exactly what `IntentClassification` expects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ARTIFACT_ZIP = '/content/drive/MyDrive/banking-ai-agent/intent_artifacts.zip'
PROJECT_ROOT = '/content/banking-ai-agent'

import os, zipfile
assert os.path.exists(ARTIFACT_ZIP), f'Missing {ARTIFACT_ZIP} on Drive.'

with zipfile.ZipFile(ARTIFACT_ZIP) as zf:
    zf.extractall('/tmp/_intent')

import shutil
for sub in ('outputs', 'sample_data', 'configs'):
    src = f'/tmp/_intent/intent_artifacts/{sub}'
    dst = f'{PROJECT_ROOT}/{sub}'
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.move(src, dst)

!ls -lh /content/banking-ai-agent/outputs | head
!wc -l /content/banking-ai-agent/sample_data/labels.txt

## Cell 4 — Smoke-test the intent classifier

Loads the LoRA adapter once and runs a few predictions. This also lets us
check VRAM usage before stacking `gpt-oss:20b` on top.

In [ ]:
%cd /content/banking-ai-agent
import sys
sys.path.insert(0, '.')

from app.nodes._lab2_inference import IntentClassification

clf = IntentClassification('configs/inference.yaml')
for msg in [
    'I am still waiting on my card?',
    'I lost my card on the train today',
    'Does my card work with Apple Pay?',
    'My transfer to John failed this morning',
]:
    print(f'\n→ {msg}')
    print('   intent =', clf(msg))

In [ ]:
# Quick VRAM check after loading the intent model.
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## Cell 5 — Launch the FastAPI server on port 8000

Starts `app.main:app` in the background. Watch `server.log` to verify the
Unsloth model loaded once at startup.

In [ ]:
%cd /content/banking-ai-agent
import os

os.environ['INTENT_MODE']         = 'unsloth'
os.environ['MOCK_LLM']            = '0'
os.environ['OLLAMA_URL']          = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']        = OLLAMA_MODEL
os.environ['INTENT_CONFIG_PATH']  = '/content/banking-ai-agent/configs/inference.yaml'
os.environ['INTENT_LABELS_PATH']  = '/content/banking-ai-agent/sample_data/labels.txt'
os.environ['PORT']                = '8000'

!pkill -f 'python run.py' 2>/dev/null; sleep 1
!nohup python run.py > server.log 2>&1 &

import time, requests
for _ in range(40):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.ok:
            print('FastAPI ready →', r.json())
            break
    except Exception:
        pass
    time.sleep(3)
else:
    print('Server did not come up in time; tail of server.log:')
    !tail -40 server.log

## Cell 6 — Open a Pinggy tunnel (manual)

Pinggy requires an interactive SSH session. Open a Colab **terminal** (View →
Terminal) and run the command below. The terminal will print a public URL of
the form `https://<id>.a.free.pinggy.link`. **Right-click → Copy** the URL —
pressing `Ctrl+C` kills the tunnel.

```bash
ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io
```

Then paste the URL into the next cell.

In [ ]:
PUBLIC_URL = ''  # ← paste the Pinggy URL here, e.g. 'https://abcd1.a.free.pinggy.link'

import requests
assert PUBLIC_URL, 'Set PUBLIC_URL first.'
print(requests.get(f'{PUBLIC_URL}/health', timeout=10).json())

## Cell 7 — Demo all sample requests

Loops over `examples/sample_requests.json` and pretty-prints the workflow
trace. Use either the local URL (`http://localhost:8000`) or the public
Pinggy URL — both work.

In [ ]:
import json, requests, textwrap

TARGET = 'http://localhost:8000'   # or PUBLIC_URL for the tunnel
examples = json.load(open('/content/banking-ai-agent/examples/sample_requests.json'))

for ex in examples:
    r = requests.post(f'{TARGET}/process', json={'message': ex['message']}, timeout=180)
    data = r.json()
    print('=' * 80)
    print('💬', ex['message'])
    print('   intent   :', data['trace']['intent']['intent'], '(' + data['trace']['intent']['source'] + ')')
    print('   priority :', data['trace']['priority']['level'])
    print('   policy   :', data['trace']['policy']['policy_id'])
    print('   action   :', data['decision']['action'])
    print('   reply    :')
    print(textwrap.indent(data['final_response'], '              '))
    print('   latency  :', data['extra'].get('latency_ms'), 'ms')

## Cell 8 — Interactive query (optional)

Useful for the video demo: change the message and re-run the cell.

In [ ]:
import requests, json
msg = 'I lost my card on the bus this morning, please help!'
r = requests.post('http://localhost:8000/process', json={'message': msg}, timeout=180)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))